In [85]:
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import numpy as np
import investpy

In [ ]:
def _price_series(df: pd.DataFrame) -> pd.Series:
    """Return a single price series from yfinance download result.
    Prefers 'Adj Close', falls back to 'Close'. Handles MultiIndex columns.
    """
    if df is None or df.empty:
        return pd.Series(dtype=float)
    # If MultiIndex columns (when multiple tickers or OHLCV levels)
    if isinstance(df.columns, pd.MultiIndex):
        # Prefer level 0 name 'Adj Close' or last level
        try:
            s = df['Adj Close'].squeeze()
            if isinstance(s, pd.DataFrame):
                # if still DataFrame (multiple tickers), take the first column
                s = s.iloc[:, 0]
            return s
        except KeyError:
            pass
        try:
            s = df['Close'].squeeze()
            if isinstance(s, pd.DataFrame):
                s = s.iloc[:, 0]
            return s
        except KeyError:
            pass
        # Fallback: try the first available price-like column
        for candidate in ['Close', 'Adj Close', 'Price']:
,   

In [2]:
# Define date range - past 5 years
end_date = datetime.now()
start_date = end_date - timedelta(days=5*365)

In [3]:
def get_sp500_return(start_date, end_date):
    """Return S&P 500 daily close price and percentage returns."""
    df = yf.download('^GSPC', start=start_date, end=end_date, progress=False, auto_adjust=True)
    # Prefer Close column and flatten if needed
    price = df['Close'].squeeze()
    returns = price.pct_change()
    return pd.DataFrame({'sp500_price': price, 'sp500_return': returns})

In [4]:
get_sp500_return(start_date, end_date)

,sp500_price,sp500_return
Date,,
2020-12-15,3694.620117,NaN
2020-12-16,3701.169922,0.001773
2020-12-17,3722.479980,0.005758
2020-12-18,3709.409912,-0.003511
2020-12-21,3694.919922,-0.003906
...,...,...
2025-12-08,6846.509766,-0.003477
2025-12-09,6840.509766,-0.000876
2025-12-10,6886.680176,0.006750


In [5]:
def get_sector_etf_returns(start_date, end_date):
    """Get sector ETF returns for XLK (Tech), XLF (Financials), XLE (Energy)"""
    tickers = ['XLK','XLF','XLE']
    data = yf.download(tickers, start=start_date, end=end_date, progress=False, auto_adjust=True)
    sector_data = pd.DataFrame()
    for t, name in zip(tickers, ['tech', 'financials', 'energy']):
        # Handle MultiIndex when multiple tickers
        price = data['Close'][t]
        sector_data[f'xlk_{name}_price' if t == 'XLK' else f'xlf_{name}_price' if t == 'XLF' else f'xle_{name}_price'] = price
        sector_data[f'xlk_{name}_return' if t == 'XLK' else f'xlf_{name}_return' if t == 'XLF' else f'xle_{name}_return'] = price.pct_change()
    return sector_data

In [6]:
get_sector_etf_returns(start_date, end_date)

,xlk_tech_price,xlk_tech_return,xlf_financials_price,xlf_financials_return,xle_energy_price,xle_energy_return
Date,,,,,,
2020-12-15,60.937439,NaN,26.192345,NaN,16.666445,NaN
2020-12-16,61.350636,0.006781,26.238115,0.001747,16.584061,-0.004943
2020-12-17,61.835907,0.007910,26.311359,0.002792,16.505795,-0.004719
2020-12-18,61.619705,-0.003496,26.082483,-0.008699,16.229811,-0.016720
2020-12-21,61.681343,0.001000,26.434078,0.013480,15.916732,-0.019290
...,...,...,...,...,...,...
2025-12-08,147.630005,0.007026,53.480000,-0.003726,45.419998,-0.010889
2025-12-09,148.020004,0.002642,53.279999,-0.003740,45.700001,0.006165
2025-12-10,148.729996,0.004797,53.889999,0.011449,46.180000,0.010503


In [7]:
def get_oil_price(start_date, end_date):
    """Get WTI crude oil price and returns"""
    oil = yf.download('CL=F', start=start_date, end=end_date, progress=False, auto_adjust=True)
    price = oil['Close'].squeeze()
    return pd.DataFrame({'oil_price': price, 'oil_return': price.pct_change()})

In [8]:
get_oil_price(start_date, end_date)

,oil_price,oil_return
Date,,
2020-12-15,47.619999,NaN
2020-12-16,47.820000,0.004200
2020-12-17,48.360001,0.011292
2020-12-18,49.099998,0.015302
2020-12-21,47.740002,-0.027699
...,...,...
2025-12-08,58.880001,-0.019973
2025-12-09,58.250000,-0.010700
2025-12-10,58.459999,0.003605


In [9]:
def get_gold_price(start_date, end_date):
    """Get gold price and returns"""
    gold = yf.download('GC=F', start=start_date, end=end_date, progress=False, auto_adjust=True)
    price = gold['Close'].squeeze()
    return pd.DataFrame({'gold_price': price, 'gold_return': price.pct_change()})

In [10]:
get_gold_price(start_date, end_date)

,gold_price,gold_return
Date,,
2020-12-15,1852.300049,NaN
2020-12-16,1856.099976,0.002051
2020-12-17,1887.199951,0.016756
2020-12-18,1885.699951,-0.000795
2020-12-21,1879.199951,-0.003447
...,...,...
2025-12-08,4187.200195,-0.006100
2025-12-09,4206.700195,0.004657
2025-12-10,4196.399902,-0.002449


In [11]:
def get_usd_index(start_date, end_date):
    """Get US Dollar Index (DXY)"""
    usd = yf.download('DX-Y.NYB', start=start_date, end=end_date, progress=False, auto_adjust=True)
    price = usd['Close'].squeeze()
    return pd.DataFrame({'usd_index': price, 'usd_return': price.pct_change()})

In [12]:
get_usd_index(start_date, end_date)

,usd_index,usd_return
Date,,
2020-12-15,90.470001,NaN
2020-12-16,90.449997,-0.000221
2020-12-17,89.820000,-0.006965
2020-12-18,90.019997,0.002227
2020-12-21,90.040001,0.000222
...,...,...
2025-12-08,99.089996,0.001010
2025-12-09,99.220001,0.001312
2025-12-10,98.790001,-0.004334


In [13]:
def get_btc_correlation(start_date, end_date, ticker='^GSPC', window=30):
    """Compute BTC returns and rolling correlation with a given stock/Index.
    Returns data only for dates when both BTC and the stock traded.
    
    Parameters:
    - start_date, end_date: date range
    - ticker: stock or index ticker (e.g., 'AAPL', '^GSPC')
    - window: rolling window size in days
    
    Returns:
    - DataFrame with columns: btc_price, btc_return, btc_<ticker>_corr_<window>d
    """
    # Fetch extra days before start_date to have enough data for rolling window
    # Need more buffer for business days: ~50 calendar days for 30 business days
    extended_start = start_date - timedelta(days=int(window * 1.5) + 10)
    
    btc = yf.download('BTC-USD', start=extended_start, end=end_date, progress=False, auto_adjust=True)
    stock = yf.download(ticker, start=extended_start, end=end_date, progress=False, auto_adjust=True)
    
    btc_price = btc['Close'].squeeze()
    stock_price = stock['Close'].squeeze()
    
    btc_ret = btc_price.pct_change()
    stock_ret = stock_price.pct_change()
    
    # Align on common dates (intersection of both trading days)
    ret_df = pd.DataFrame({'btc_ret': btc_ret, 'stock_ret': stock_ret}).dropna()
    
    # Compute rolling correlation on aligned returns
    ret_df['corr'] = ret_df['btc_ret'].rolling(window=window, min_periods=window).corr(ret_df['stock_ret'])
    
    # Get BTC price for the aligned dates
    ret_df['btc_price'] = btc_price.reindex(ret_df.index)
    
    # Trim back to original date range
    ret_df = ret_df.loc[start_date:]
    
    corr_col = f"btc_{ticker.replace('^','').lower()}_corr_{window}d"
    return ret_df[['btc_price', 'btc_ret', 'corr']].rename(columns={'btc_ret': 'btc_return', 'corr': corr_col})

In [14]:
get_btc_correlation(start_date, end_date, ticker='^GSPC', window=30).tail()

,btc_price,btc_return,btc_gspc_corr_30d
Date,,,
2025-12-08,90640.203125,0.002595,0.467395
2025-12-09,92691.710938,0.022634,0.459188
2025-12-10,92020.945312,-0.007237,0.456084
2025-12-11,92511.335938,0.005329,0.461064
2025-12-12,90270.414062,-0.024223,0.471716


In [58]:
def get_treasury_yields(start_date, end_date):
    """Get Treasury yields for 2-year, 5-year, and 10-year"""
    treasury_data = pd.DataFrame()
    # 2Y: try ^2YY, fallback to ^UST2Y (if available)
    for t, col in [('^IRX','treasury_3m_yld_idx'), ('2YY=F','treasury_2y_yld'), ('ZT=F','treasury_2y_price'), ('^FVX','treasury_5y_yld_idx'), ('ZF=F','treasury_5y_price'), ('^TNX','treasury_10y_yld_idx'), ('ZN=F','treasury_10y_price')]:
        try:
            df = yf.download(t, start=start_date, end=end_date, progress=False, auto_adjust=True)
            if not df.empty:
                if 'Close' in df.columns:
                    price = df['Close']
                else:
                    price = df.squeeze()
                treasury_data[col] = price
        except Exception:
            pass
    return treasury_data

In [59]:
get_treasury_yields(start_date, end_date)

,treasury_3m_yld_idx,treasury_2y_yld,treasury_2y_price,treasury_5y_yld_idx,treasury_5y_price,treasury_10y_yld_idx,treasury_10y_price
Date,,,,,,,
2020-12-15,0.070,NaN,110.472656,0.375,125.710938,0.923,138.296875
2020-12-16,0.075,NaN,110.472656,0.370,125.742188,0.920,138.328125
2020-12-17,0.080,NaN,110.464844,0.377,125.703125,0.930,138.296875
2020-12-18,0.080,NaN,110.468750,0.381,125.695312,0.948,138.203125
2020-12-21,0.078,NaN,110.468750,0.383,125.679688,0.941,138.406250
...,...,...,...,...,...,...,...
2025-12-08,3.618,3.43,104.117188,3.753,109.054688,4.172,112.390625
2025-12-09,3.632,3.43,104.058594,3.780,108.921875,4.186,112.234375
2025-12-10,3.593,3.43,104.156250,3.757,109.046875,4.164,112.359375


In [61]:
def get_credit_spreads(start_date, end_date):
    """Get High-Yield and Investment-Grade credit spreads (ETF proxy)"""
    data = yf.download(['HYG','LQD','AGG'], start=start_date, end=end_date, progress=False, auto_adjust=True)
    if isinstance(data.columns, pd.MultiIndex):
        hyg = data['Close']['HYG']
        lqd = data['Close']['LQD']
        agg = data['Close']['AGG']
    else:
        # Rare case when MultiIndex not returned; fetch individually
        hyg = yf.download('HYG', start=start_date, end=end_date, progress=False, auto_adjust=True)['Close']
        lqd = yf.download('LQD', start=start_date, end=end_date, progress=False, auto_adjust=True)['Close']
        agg = yf.download('AGG', start=start_date, end=end_date, progress=False, auto_adjust=True)['Close']
    hy_ret = hyg.pct_change()
    ig_ret = lqd.pct_change()
    agg_ret = agg.pct_change()
    hy_spread = (hy_ret - agg_ret).cumsum()
    ig_spread = (ig_ret - agg_ret).cumsum()
    return pd.DataFrame({'hy_spread': hy_spread, 'ig_spread': ig_spread, 'hy_return': hy_ret, 'ig_return': ig_ret})

In [62]:
get_credit_spreads(start_date, end_date)

/var/folders/7c/kv_rdjps3ts1c7_5mb3qszw40000gn/T/ipykernel_37039/4207648757.py:14: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ig_ret = lqd.pct_change()


,hy_spread,ig_spread,hy_return,ig_return
Date,,,,
2020-12-15,NaN,NaN,NaN,NaN
2020-12-16,-0.001071,-0.000377,-0.001495,-0.000800
2020-12-17,0.000463,0.001126,0.001492,0.001460
2020-12-18,0.001780,0.000906,0.000808,-0.000728
2020-12-21,-0.001797,-0.001063,-0.003576,-0.001969
...,...,...,...,...
2025-12-08,0.216882,-0.004029,-0.002477,-0.002075
2025-12-09,0.216666,-0.004393,-0.001117,-0.001265
2025-12-10,0.216815,-0.002984,0.003356,0.004616


In [83]:
def get_fx_pairs(start_date, end_date, fx_pairs):
    """
    Get daily prices for a list of foreign exchange pairs.
    Falls back to investpy if Yahoo Finance doesn't have the data.
    
    Parameters:
    - start_date: Start date for data collection
    - end_date: End date for data collection
    - fx_pairs: List of FX pairs in format like "EUR/USD", "USD/JPY", etc.
    
    Returns:
    - DataFrame with daily prices for all FX pairs
    """
    import investpy
    
    fx_data = pd.DataFrame()
    
    for pair in fx_pairs:
        # Try Yahoo Finance first
        ticker = pair.replace("/", "") + "=X"
        col_name = pair.replace("/", "_").lower()
        
        try:
            df = yf.download(ticker, start=start_date, end=end_date, progress=False, auto_adjust=True)
            if not df.empty and len(df) > 10:  # Check if we have meaningful data
                if 'Close' in df.columns:
                    price = df['Close'].squeeze()
                else:
                    price = df.squeeze()
                fx_data[f'{col_name}_price'] = price
                fx_data[f'{col_name}_return'] = price.pct_change()
                print(f"✓ {pair} fetched from Yahoo Finance")
            else:
                raise ValueError(f"Insufficient data from Yahoo Finance for {pair}")
        except Exception as yf_error:
            # Fallback to investpy
            try:
                print(f"⚠ Yahoo Finance failed for {pair}, trying investpy...")
                # Format dates for investpy (DD/MM/YYYY)
                from_date = start_date.strftime('%d/%m/%Y')
                to_date = end_date.strftime('%d/%m/%Y')
                
                df_investpy = investpy.get_currency_cross_historical_data(
                    currency_cross=pair,
                    from_date=from_date,
                    to_date=to_date
                )
                
                if not df_investpy.empty:
                    price = df_investpy['Close']
                    fx_data[f'{col_name}_price'] = price
                    fx_data[f'{col_name}_return'] = price.pct_change()
                    print(f"✓ {pair} fetched from investpy")
                else:
                    print(f"✗ No data available for {pair} from either source")
            except Exception as inv_error:
                print(f"✗ Error fetching {pair} from both sources:")
                print(f"  Yahoo Finance: {yf_error}")
                print(f"  investpy: {inv_error}")
    
    return fx_data

In [94]:
# Example usage with your FX pairs
fx_pairs = [
    "EUR/USD",
    "USD/JPY",
    "USD/CNH",
    "USD/CNY",
    "USD/KRW",
    "USD/TWD",
    "USD/INR",
    "USD/BRL",
    "USD/MXN",
    "USD/CAD",
    "USD/CLP",
    "AUD/USD",
    "USD/NOK",
    "USD/RUB",
    "USD/GBP",
    "USD/CHF"
]


get_fx_pairs(start_date, end_date, fx_pairs)


1 Failed download:
['USDCNH=X']: YFPricesMissingError('possibly delisted; no price data found  (1d 2020-12-15 17:20:53.142160 -> 2025-12-14 17:20:53.142160)')


✓ EUR/USD fetched from Yahoo Finance
✓ USD/JPY fetched from Yahoo Finance
⚠ Yahoo Finance failed for USD/CNH, trying investpy...
✓ USD/CNH fetched from investpy
✓ USD/CNY fetched from Yahoo Finance
✓ USD/KRW fetched from Yahoo Finance
✓ USD/TWD fetched from Yahoo Finance
✓ USD/INR fetched from Yahoo Finance
✓ USD/BRL fetched from Yahoo Finance
✓ USD/MXN fetched from Yahoo Finance
✓ USD/CAD fetched from Yahoo Finance
✓ USD/CLP fetched from Yahoo Finance
✓ AUD/USD fetched from Yahoo Finance
✓ USD/NOK fetched from Yahoo Finance
✓ USD/RUB fetched from Yahoo Finance
✓ USD/GBP fetched from Yahoo Finance
✓ USD/CHF fetched from Yahoo Finance


,eur_usd_price,eur_usd_return,usd_jpy_price,usd_jpy_return,usd_cnh_price,usd_cnh_return,usd_cny_price,usd_cny_return,usd_krw_price,usd_krw_return,...,aud_usd_price,aud_usd_return,usd_nok_price,usd_nok_return,usd_rub_price,usd_rub_return,usd_gbp_price,usd_gbp_return,usd_chf_price,usd_chf_return
Date,,,,,,,,,,,,,,,,,,,,,
2020-12-15,1.214890,NaN,104.010002,NaN,6.5162,NaN,6.5497,NaN,1092.060059,NaN,...,0.753920,NaN,8.72890,NaN,74.121300,NaN,0.750140,NaN,0.886580,NaN
2020-12-16,1.215430,0.000445,103.634003,-0.003615,6.5111,-0.000783,6.5383,-0.001740,1088.209961,-0.003526,...,0.755561,0.002176,8.71787,-0.001264,73.278503,-0.011371,0.743800,-0.008452,0.885440,-0.001286
2020-12-17,1.219959,0.003726,103.474998,-0.001534,6.5165,0.000829,6.5313,-0.001071,1091.650024,0.003161,...,0.757000,0.001905,8.66650,-0.005892,73.347900,0.000947,0.740580,-0.004329,0.885390,-0.000056
2020-12-18,1.226272,0.005175,103.130997,-0.003324,6.5182,0.000261,6.5321,0.000123,1092.160034,0.000467,...,0.761530,0.005984,8.56571,-0.011630,73.090401,-0.003511,0.736910,-0.004956,0.884720,-0.000757
2020-12-21,1.221613,-0.003799,103.457001,0.003161,6.5332,0.002301,6.5360,0.000597,1098.979980,0.006244,...,0.758743,-0.003660,8.63030,0.007540,73.481003,0.005344,0.746160,0.012552,0.885480,0.000859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-08,1.164022,-0.000221,155.339996,0.001276,7.0716,0.000382,7.0696,-0.000255,1471.810059,-0.000387,...,0.663540,0.004387,10.10922,-0.000532,75.990913,-0.000077,0.750492,-0.000211,0.804850,0.001331
2025-12-09,1.164144,0.000105,155.843994,0.003244,7.0609,-0.001513,7.0710,0.000198,1468.510010,-0.002242,...,0.662730,-0.001221,10.12247,0.001311,76.545464,0.007298,0.750368,-0.000165,0.806638,0.002222
2025-12-10,1.162831,-0.001128,156.837997,0.006378,7.0608,-0.000014,7.0633,-0.001089,1467.930054,-0.000395,...,0.664099,0.002065,10.14630,0.002354,77.186211,0.008371,0.751680,0.001748,0.805900,-0.000915


In [90]:
fx_pair_to_sectors = {
    "EUR/USD": [
        "Information Technology",
        "Communication Services",
        "Consumer Discretionary",
        "Consumer Staples",
        "Industrials",
        "Financials",
        "Health Care",
        "Real Estate"
    ],
    "USD/JPY": [
        "Information Technology",
        "Communication Services",
        "Consumer Discretionary",
        "Industrials",
        "Financials",
        "Health Care"
    ],
    "USD/CNH": [
        "Information Technology",
        "Consumer Discretionary",
        "Industrials",
        "Health Care",
        "Financials"
    ],
    "USD/CNY": [
        "Industrials",
        "Materials",
        "Financials"
    ],
    "USD/KRW": [
        "Information Technology"
    ],
    "USD/TWD": [
        "Information Technology"
    ],
    "USD/INR": [
        "Communication Services",
        "Consumer Staples"
    ],
    "USD/BRL": [
        "Communication Services",
        "Consumer Staples",
        "Materials"
    ],
    "USD/MXN": [
        "Consumer Discretionary",
        "Industrials",
        "Energy",
        "Consumer Staples"
    ],
    "USD/CAD": [
        "Materials",
        "Energy",
        "Utilities"
    ],
    "USD/CLP": [
        "Materials"
    ],
    "AUD/USD": [
        "Materials"
    ],
    "USD/NOK": [
        "Energy"
    ],
    "USD/RUB": [
        "Energy"
    ],
    "USD/GBP": [
        "Financials"
    ],
    "USD/CHF": [
        "Health Care"
    ]
}


In [ ]:
def get_fx_sector_flags(fx_pairs, fx_pair_to_sectors):
    """
    Create a dataframe with sector flags for each FX pair.
    
    Parameters:
    - fx_pairs: List of FX pairs (e.g., ["EUR/USD", "USD/JPY"])
    - fx_pair_to_sectors: Dictionary mapping FX pairs to their relevant sectors
    
    Returns:
    - DataFrame where rows are FX pairs and columns are sectors with binary flags (1 if relevant, 0 otherwise)
    """
    # Get all unique sectors
    all_sectors = sorted(set(sector for sectors in fx_pair_to_sectors.values() for sector in sectors))
    
    # Create the dataframe
    sector_flags = pd.DataFrame(0, index=fx_pairs, columns=all_sectors)
    
    # Fill in the flags
    for fx_pair in fx_pairs:
        if fx_pair in fx_pair_to_sectors:
            for sector in fx_pair_to_sectors[fx_pair]:
                sector_flags.loc[fx_pair, sector] = 1
    
    return sector_flags

In [ ]:
# Test the function
get_fx_sector_flags(fx_pairs, fx_pair_to_sectors)

,Communication Services,Consumer Discretionary,Consumer Staples,Energy,Financials,Health Care,Industrials,Information Technology,Materials,Real Estate,Utilities
EUR/USD,1,1,1,0,1,1,1,1,0,1,0
USD/JPY,1,1,0,0,1,1,1,1,0,0,0
USD/CNH,0,1,0,0,1,1,1,1,0,0,0
USD/CNY,0,0,0,0,1,0,1,0,1,0,0
USD/KRW,0,0,0,0,0,0,0,1,0,0,0
USD/TWD,0,0,0,0,0,0,0,1,0,0,0
USD/INR,1,0,1,0,0,0,0,0,0,0,0
USD/BRL,1,0,1,0,0,0,0,0,1,0,0
USD/MXN,0,1,1,1,0,0,1,0,0,0,0
USD/CAD,0,0,0,1,0,0,0,0,1,0,1


In [95]:
def get_all_cross_asset_features(start_date, end_date, fx_pairs=None):
    """
    Consolidate all cross-asset features into a single dataframe
    
    Parameters:
    - start_date: Start date for data collection
    - end_date: End date for data collection
    - fx_pairs: Optional list of FX pairs to include (e.g., ["EUR/USD", "USD/JPY"])
    
    Returns:
    - DataFrame with all cross-asset features
    """
    print("Fetching S&P 500 returns...")
    df_sp500 = get_sp500_return(start_date, end_date)
    
    print("Fetching sector ETF returns...")
    df_sectors = get_sector_etf_returns(start_date, end_date)
    
    print("Fetching oil price...")
    df_oil = get_oil_price(start_date, end_date)
    
    print("Fetching gold price...")
    df_gold = get_gold_price(start_date, end_date)
    
    print("Fetching USD index...")
    df_usd = get_usd_index(start_date, end_date)
    
    print("Fetching BTC correlation...")
    df_btc = get_btc_correlation(start_date, end_date)
    
    print("Fetching treasury yields...")
    df_treasuries = get_treasury_yields(start_date, end_date)
    
    print("Fetching credit spreads...")
    df_credit = get_credit_spreads(start_date, end_date)
    
    # List of dataframes to concatenate
    dfs_to_concat = [
        df_sp500,
        df_sectors,
        df_oil,
        df_gold,
        df_usd,
        df_btc,
        df_treasuries,
        df_credit
    ]
    
    # Fetch FX pairs if provided
    if fx_pairs:
        print("\nFetching FX pairs...")
        df_fx = get_fx_pairs(start_date, end_date, fx_pairs)
        dfs_to_concat.append(df_fx)
    
    # Concatenate all dataframes
    print("\nCombining all features...")
    cross_asset_df = pd.concat(dfs_to_concat, axis=1)
    
    print(f"\nFinal dataset shape: {cross_asset_df.shape}")
    print(f"Date range: {cross_asset_df.index.min()} to {cross_asset_df.index.max()}")
    print(f"\nFeatures included: {list(cross_asset_df.columns)}")
    
    return cross_asset_df

In [96]:
# Test with FX pairs included
get_all_cross_asset_features(start_date, end_date, fx_pairs=fx_pairs)

Fetching S&P 500 returns...
Fetching sector ETF returns...
Fetching oil price...
Fetching gold price...
Fetching USD index...
Fetching BTC correlation...
Fetching treasury yields...


/var/folders/7c/kv_rdjps3ts1c7_5mb3qszw40000gn/T/ipykernel_37039/4207648757.py:14: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ig_ret = lqd.pct_change()

1 Failed download:
['USDCNH=X']: YFPricesMissingError('possibly delisted; no price data found  (1d 2020-12-15 17:20:53.142160 -> 2025-12-14 17:20:53.142160)')
['USDCNH=X']: YFPricesMissingError('possibly delisted; no price data found  (1d 2020-12-15 17:20:53.142160 -> 2025-12-14 17:20:53.142160)')


Fetching credit spreads...

Fetching FX pairs...
✓ EUR/USD fetched from Yahoo Finance
✓ USD/JPY fetched from Yahoo Finance
⚠ Yahoo Finance failed for USD/CNH, trying investpy...
✓ USD/CNH fetched from investpy
✓ USD/CNY fetched from Yahoo Finance
✓ USD/KRW fetched from Yahoo Finance
✓ USD/TWD fetched from Yahoo Finance
✓ USD/INR fetched from Yahoo Finance
✓ USD/BRL fetched from Yahoo Finance
✓ USD/MXN fetched from Yahoo Finance
✓ USD/CAD fetched from Yahoo Finance
✓ USD/CLP fetched from Yahoo Finance
✓ AUD/USD fetched from Yahoo Finance
✓ USD/NOK fetched from Yahoo Finance
✓ USD/RUB fetched from Yahoo Finance
✓ USD/GBP fetched from Yahoo Finance
✓ USD/CHF fetched from Yahoo Finance

Combining all features...

Final dataset shape: (1302, 60)
Date range: 2020-12-15 00:00:00 to 2025-12-12 00:00:00

Features included: ['sp500_price', 'sp500_return', 'xlk_tech_price', 'xlk_tech_return', 'xlf_financials_price', 'xlf_financials_return', 'xle_energy_price', 'xle_energy_return', 'oil_price', 'o

,sp500_price,sp500_return,xlk_tech_price,xlk_tech_return,xlf_financials_price,xlf_financials_return,xle_energy_price,xle_energy_return,oil_price,oil_return,...,aud_usd_price,aud_usd_return,usd_nok_price,usd_nok_return,usd_rub_price,usd_rub_return,usd_gbp_price,usd_gbp_return,usd_chf_price,usd_chf_return
Date,,,,,,,,,,,,,,,,,,,,,
2020-12-15,3694.620117,NaN,60.937439,NaN,26.192341,NaN,16.666445,NaN,47.619999,NaN,...,0.753920,NaN,8.72890,NaN,74.121300,NaN,0.750140,NaN,0.886580,NaN
2020-12-16,3701.169922,0.001773,61.350639,0.006781,26.238115,0.001748,16.584055,-0.004943,47.820000,0.004200,...,0.755561,0.002176,8.71787,-0.001264,73.278503,-0.011371,0.743800,-0.008452,0.885440,-0.001286
2020-12-17,3722.479980,0.005758,61.835903,0.007910,26.311356,0.002791,16.505795,-0.004719,48.360001,0.011292,...,0.757000,0.001905,8.66650,-0.005892,73.347900,0.000947,0.740580,-0.004329,0.885390,-0.000056
2020-12-18,3709.409912,-0.003511,61.619705,-0.003496,26.082481,-0.008699,16.229807,-0.016721,49.099998,0.015302,...,0.761530,0.005984,8.56571,-0.011630,73.090401,-0.003511,0.736910,-0.004956,0.884720,-0.000757
2020-12-21,3694.919922,-0.003906,61.681335,0.001000,26.434080,0.013480,15.916729,-0.019290,47.740002,-0.027699,...,0.758743,-0.003660,8.63030,0.007540,73.481003,0.005344,0.746160,0.012552,0.885480,0.000859
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-08,6846.509766,-0.003477,147.630005,0.007026,53.480000,-0.003726,45.419998,-0.010889,58.880001,-0.019973,...,0.663540,0.004387,10.10922,-0.000532,75.990913,-0.000077,0.750492,-0.000211,0.804850,0.001331
2025-12-09,6840.509766,-0.000876,148.020004,0.002642,53.279999,-0.003740,45.700001,0.006165,58.250000,-0.010700,...,0.662730,-0.001221,10.12247,0.001311,76.545464,0.007298,0.750368,-0.000165,0.806638,0.002222
2025-12-10,6886.680176,0.006750,148.729996,0.004797,53.889999,0.011449,46.180000,0.010503,58.459999,0.003605,...,0.664099,0.002065,10.14630,0.002354,77.186211,0.008371,0.751680,0.001748,0.805900,-0.000915
